# NOUS — Neural Omnidimensional Unified System
**Training on Colab GPU: computation as physical relaxation, no BPTT.**

| Config | Params | Device | Dataset | Target tokens |
|--------|--------|--------|---------|---------------|
| NOUS-Small | 44M | T4 / V100 | WikiText-103 | 100M |
| NOUS-7B | 6.67B | A100 80GB | C4 / RedPajama | 1B+ |

Checkpoints saved to Google Drive automatically.

In [ ]:
# ── Cell 1: GPU check ─────────────────────────────────────────────────────
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# ── Cell 2: Mount Drive (checkpoints persist across sessions) ─────────────
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/nous_checkpoints'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoints → {CKPT_DIR}")

In [ ]:
# ── Cell 3: Install deps + clone repo ─────────────────────────────────────
!pip install -q transformers datasets accelerate sentencepiece
!git clone https://github.com/HemanthKiranPolu/nous.git /content/nous
%cd /content/nous
!git checkout scale/nous-7b
!git log --oneline -3

In [ ]:
# ── Cell 4: Config — choose model size and dataset ────────────────────────
import sys; sys.path.insert(0, '/content/nous')

# ↓ Change these
MODEL_SIZE  = 'small'        # 'small' (44M, fits any GPU) | '7b' (6.67B, needs A100)
DATASET     = 'wikitext103'  # 'wikitext103' | 'c4' | 'wikitext2'
MAX_TOKENS  = 100_000_000    # 100M for serious small run; 1B+ for 7B
BATCH_SIZE  = 16             # T4: 8 | V100: 16 | A100: 32
ACCUM_STEPS = 4

from nous.nous7b.config import NOUS_SMALL, NOUS_7B
cfg = NOUS_SMALL if MODEL_SIZE == 'small' else NOUS_7B
print(f"Config: {MODEL_SIZE.upper()}")
print(f"Dataset: {DATASET}  max_tokens: {MAX_TOKENS/1e6:.0f}M")

In [ ]:
# ── Cell 5: Build model ────────────────────────────────────────────────────
import torch
from dataclasses import replace
from transformers import GPT2Tokenizer
from nous.nous7b.energy_net_7b import NOUSEnergyNet7B
from nous.nous7b.ode_batched import BatchedEulerODE

device = torch.device('cuda')
dtype  = torch.bfloat16 if MODEL_SIZE == '7b' else torch.float32

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
cfg = replace(cfg, vocab_size=tokenizer.vocab_size)

model = NOUSEnergyNet7B(cfg).to(device=device, dtype=dtype)
model.parameter_summary()

# Compile for 2-3× speedup on A100 (PyTorch 2.0+)
if torch.__version__ >= '2.0' and MODEL_SIZE == '7b':
    model = torch.compile(model)
    print('torch.compile enabled')

In [ ]:
# ── Cell 6: Resume from checkpoint (optional) ─────────────────────────────
import glob, os

RESUME = True   # set False to start fresh

opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                         weight_decay=cfg.weight_decay, betas=(0.9, 0.95))
start_tokens = 0

if RESUME:
    ckpts = sorted(glob.glob(f'{CKPT_DIR}/step_*.pt'))
    if ckpts:
        ckpt = torch.load(ckpts[-1], map_location=device)
        model.load_state_dict(ckpt['model'])
        opt.load_state_dict(ckpt['opt'])
        start_tokens = ckpt.get('total_tokens', 0)
        print(f"Resumed from {ckpts[-1]}  ({start_tokens/1e6:.1f}M tokens seen)")
    else:
        print('No checkpoint found — starting fresh')
else:
    print('Starting fresh')

In [ ]:
# ── Cell 7: Training loop ──────────────────────────────────────────────────
import math, time, numpy as np
import torch.nn.functional as F
from datasets import load_dataset
from nous.nous7b.ode_batched import BatchedEulerODE

ode = BatchedEulerODE(model, cfg)

# ── Data stream ─────────────────────────────────────────────────────────
def sentence_stream(dataset_name, tokenizer, min_len=4, max_len=64):
    if dataset_name == 'wikitext103':
        ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train')
        texts = (item['text'] for item in ds)
    elif dataset_name == 'c4':
        ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)
        texts = (item['text'] for item in ds)
    else:  # wikitext2
        ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
        texts = (item['text'] for item in ds)
    for text in texts:
        t = text.strip()
        if not t or t.startswith('='): continue
        ids = tokenizer.encode(t)
        for start in range(0, max(1, len(ids) - min_len), max_len // 2):
            chunk = ids[start:start + max_len]
            if len(chunk) >= min_len:
                yield chunk

def batch_stream(stream, B):
    buf = []
    for s in stream:
        buf.append(s)
        if len(buf) == B:
            yield buf; buf = []
    if buf: yield buf

def _param_grad(model, x_embed, q_star):
    for p in model.parameters():
        if p.grad is not None: p.grad.zero_()
    E = model(x_embed, q_star.detach())
    E.backward()
    return {n: p.grad.clone() if p.grad is not None else torch.zeros_like(p)
            for n, p in model.named_parameters()}

# ── Loop ─────────────────────────────────────────────────────────────────
stream = batch_stream(sentence_stream(DATASET, tokenizer), BATCH_SIZE)
global_step  = 0
total_tokens = start_tokens
loss_hist, morpho_total = [], 0
t0 = time.time()

print(f'\n{"Step":>7}  {"PPL":>9}  {"Morpho":>7}  {"Mtok":>7}  {"tok/s":>7}')
print('─' * 55)

for batch_idx, batch_sents in enumerate(stream):
    B_cur = len(batch_sents)
    q_states = torch.zeros(B_cur, cfg.state_dim, device=device, dtype=dtype)
    max_t = max(len(s) for s in batch_sents) - 1
    lens  = [len(s) - 1 for s in batch_sents]

    if batch_idx % ACCUM_STEPS == 0:
        opt.zero_grad()

    for t in range(max_t):
        active = [b for b in range(B_cur) if t < lens[b]]
        if not active: continue

        x_t   = torch.stack([model.embedding(torch.tensor(batch_sents[b][t], device=device)).detach()
                              for b in active]).to(dtype)
        tgt_t = torch.tensor([batch_sents[b][t+1] for b in active], device=device)
        q0_t  = q_states[active]

        q_free  = ode.solve_batch(x_t, q0_t, n_steps=cfg.n_steps_free)
        extra   = [(lambda i: lambda q: cfg.eps * F.cross_entropy(
                        model.decode(q).unsqueeze(0), tgt_t[i:i+1]))(i)
                   for i in range(len(active))]
        q_nudge = ode.solve_batch(x_t, q0_t, n_steps=cfg.n_steps_nudge, extra_fns=extra)

        scale = 1.0 / (len(active) * cfg.eps)
        step_loss = 0.0
        for i, b in enumerate(active):
            gf = _param_grad(model, x_t[i], q_free[i])
            gn = _param_grad(model, x_t[i], q_nudge[i])
            for name, param in model.named_parameters():
                if param.requires_grad and name in gf:
                    g = scale * (gn[name] - gf[name])
                    param.grad = g if param.grad is None else param.grad.add_(g)
            with torch.no_grad():
                step_loss += F.cross_entropy(model.decode(q_free[i]).unsqueeze(0),
                                              tgt_t[i:i+1]).item()
            q_states[b] = q_free[i]

        ce = sum(F.cross_entropy(model.decode(q_nudge[i]).unsqueeze(0), tgt_t[i:i+1])
                 for i in range(len(active))) / len(active)
        ce.backward()

        loss_hist.append(step_loss / len(active))
        total_tokens += len(active)

    if (batch_idx + 1) % ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        global_step += 1

    if global_step % 100 == 0 and (batch_idx + 1) % ACCUM_STEPS == 0:
        avg = np.mean(loss_hist[-400:])
        ppl = math.exp(min(avg, 20))
        tps = total_tokens / (time.time() - t0)
        print(f'{global_step:7d}  {ppl:9.1f}  {morpho_total:7d}  '
              f'{total_tokens/1e6:7.2f}  {tps:7.0f}', flush=True)

    if global_step % 1000 == 0 and global_step > 0 and (batch_idx + 1) % ACCUM_STEPS == 0:
        ckpt_path = f'{CKPT_DIR}/step_{global_step:07d}.pt'
        torch.save({'step': global_step, 'total_tokens': total_tokens,
                    'model': model.state_dict(), 'opt': opt.state_dict(),
                    'loss': float(np.mean(loss_hist[-500:]))}, ckpt_path)
        print(f'  → checkpoint: {ckpt_path}')

    if total_tokens - start_tokens >= MAX_TOKENS:
        print(f'\nDone — {(total_tokens-start_tokens)/1e6:.1f}M tokens trained.')
        break

In [ ]:
# ── Cell 8: Plot training curve ────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np, math

window = 500
ppls = [math.exp(min(np.mean(loss_hist[max(0,i-window):i+1]), 20))
        for i in range(0, len(loss_hist), 100)]

plt.figure(figsize=(10, 4))
plt.plot(ppls, color='#9060ff', lw=2)
plt.xlabel('Steps (×100 tokens)'); plt.ylabel('Perplexity')
plt.title(f'NOUS-{MODEL_SIZE.upper()} — {DATASET} — {total_tokens/1e6:.1f}M tokens')
plt.grid(alpha=0.3); plt.yscale('log')
plt.tight_layout(); plt.show()
print(f'Final PPL: {ppls[-1]:.1f}')

In [ ]:
# ── Cell 9: Inference — watch the manifold relax ──────────────────────────
prompt_text = 'The neural network learned'
prompt_ids  = tokenizer.encode(prompt_text)
max_new     = 30

model.eval()
q = torch.zeros(cfg.state_dim, device=device, dtype=dtype)

generated = list(prompt_ids)
for tok in prompt_ids[:-1]:   # warm up state
    x = model.embedding(torch.tensor(tok, device=device)).detach()
    q = ode.solve_batch(x.unsqueeze(0), q.unsqueeze(0),
                        n_steps=cfg.n_steps_free)[0]

for _ in range(max_new):
    x = model.embedding(torch.tensor(generated[-1], device=device)).detach()
    q = ode.solve_batch(x.unsqueeze(0), q.unsqueeze(0),
                        n_steps=cfg.n_steps_free)[0]
    logits = model.decode(q)
    next_tok = logits.argmax().item()
    generated.append(next_tok)
    if next_tok == tokenizer.eos_token_id:
        break

print('Output:')
print(tokenizer.decode(generated))